# Arno River Level Forecasting

Chronological evaluation of daily hydrometry against persistence and seasonal baselines. The input workbook must contain `Date`, `Actual_Hydrometry`, `Mean_Temp` and `Mean_Rainfall`. Synthetic mode checks execution only.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd() if (Path.cwd() / "arno_forecasting").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display
from arno_forecasting.data import load_data, synthetic_data
from arno_forecasting.experiment import run_experiment

## Input and forecast assumptions

Set `ARNO_DATA` to use another input path. The default run needs the original workbook in `data/`. `ARNO_DEMO=1` selects a deterministic synthetic fixture. Future weather defaults to calendar-day climatology fitted to past observations; using observed future weather is a separate conditional evaluation.

In [ ]:
DEMO = os.environ.get("ARNO_DEMO") == "1"
if DEMO:
    frame = synthetic_data()
    quality = {"source": "Synthetic installation fixture", "seed": 42}
    output = ROOT / "results/demo"
    horizon = validation_days = 90
else:
    input_path = Path(os.environ.get("ARNO_DATA", str(ROOT / "data/RIVER_TRAIN_DATASET_A1.xlsx")))
    frame, quality = load_data(input_path)
    output = ROOT / "results/reproduced"
    horizon = validation_days = 365

print("Synthetic fixture" if DEMO else "Input workbook")
print(quality)
frame.head()

## Evaluation protocol

Hold out the final forecast horizon for testing and the preceding interval for validation. Select a candidate using validation MAE, then refit it on all data preceding the test origin. Each forecast covers the full horizon without updating from held-out river levels. Annual Fourier features use a 365.25-day period; 52 daily observations would represent a 52-day cycle.

In [ ]:
manifest = run_experiment(
    frame, output, quality, horizon=horizon, validation_days=validation_days,
    weather_mode="climatology", synthetic=DEMO
)

In [ ]:
validation_scores = pd.DataFrame(manifest["validation_metrics"]).T[["mae", "rmse", "days"]]
test_scores = pd.DataFrame(manifest["test_metrics"]).T[["mae", "rmse", "days"]]
print("Selected using validation MAE:", manifest["selected_model"])
display(validation_scores)
display(test_scores)
if manifest["unavailable_models"]:
    print("Models requiring attention:", manifest["unavailable_models"])

In [ ]:
display(Image(filename=str(output / "forecast_comparison.png")))

## Interpretation

MAE and RMSE use the hydrometry column's source units. Synthetic scores are not measurements on the Arno River. Fixed-origin forecasts with historical weather patterns do not establish flood-warning performance. Input preprocessing and units must be documented separately; this workflow cannot establish how an earlier workbook was prepared. Historical model-comparison images are preserved under `docs/legacy_figures`, with their limits described there.